# Referencia- és kiterjesztett implementáció konzisztencia-ellenőrzése

Ez a notebook a **„A Crouch-féle szimulációs prevalencia-becslés kiterjesztése folytonos időbeli modellezésre”** című szakdolgozat validációs anyaga.  
A dolgozat a Crouch-féle incidencia–túlélés alapú Monte Carlo prevalenciabecslés elméleti keretére és a meglévő `rprev` referencia-implementációra épül. A központi feladat a referencia-eljárás több indexdátum együttes kezelésére szolgáló kiterjesztése, valamint ennek módszertani és numerikus validálása (lásd: 3–5. fejezet).

Ennek megfelelően a **referencia** implementáció definíció szerint egyetlen indexdátumhoz ad prevalencia-becslést, ezért több indexdátum esetén az eredmény egyindexű `prevalence()` hívások sorozatából állítható elő. A **kiterjesztett implementáció** ezzel szemben több indexdátum együttes kezelésére készült, és a teljes indexrácsra konzisztens becslést ad egy közös futási keretben (lásd: 4. fejezet – referencia implementáció és a kiterjesztés tárgya; 5. fejezet – matematikai modell és csomagszintű illesztés).

## Validációs cél

A notebook célja annak vizsgálata, hogy a referencia implementációból indexdátumonként összeállított kimenet és a natív többindexű kiterjesztett futtatás kimenete azonos bemenetek (adat, modellek), azonos indexrács és azonos további paraméterek mellett összevethető-e, továbbá hogy a megfigyelt eltérések a kiterjesztés definiált módszertani és algoritmikus sajátosságai mellett értelmezhetők-e.

## Egyezési kritériumok és eltérésértelmezés

A két implementáció futási szerkezete eltér, ezért a véletlenszám-generátor hívásainak száma és ütemezése sem azonos (lásd dolgozat 5. fejezet). A referencia-megoldás indexdátumonként külön futtatással állít elő prevalencia-becslést, így a bináris státuszok véletlen komponense is indexdátumonként, elkülönülő Bernoulli-sorsolásokon keresztül képződik. Ezzel szemben a kiterjesztett megoldásban a státuszképzés véletlen komponense esetszinten egyetlen `xi ~ Unif(0,1)` küszöbhöz kötött, és ezt hasonlítjuk össze az indexrács pontjaihoz tartozó túlélési valószínűségekkel. A két státuszképzési szervezés különbsége miatt azonos induló véletlenmag mellett sem várható bitazonos kimenet; az összevetést a kimeneti összefoglalók (pontbecslés, intervallumhatárok) szintjén végezzük.

A referencia és a kiterjesztett implementáció kimenetét akkor tekintjük konzisztensnek, ha a relatív eltérések az indexrácson kontrolláltak. A pontbecslésre a

$$
\delta_{\widehat{P}}(t_k)
=
\frac{\widehat{P}_{\mathrm{ext}}(t_k)-\widehat{P}_{\mathrm{ref}}(t_k)}{\widehat{P}_{\mathrm{ref}}(t_k)}
$$

relatív eltérést, míg az intervallumhatárokra (minden $\alpha\in\{0.025,\,0.975\}$ esetén) a

$$
\delta_{q,\alpha}(t_k)
=
\frac{q_{\alpha,\mathrm{ext}}(t_k)-q_{\alpha,\mathrm{ref}}(t_k)}{q_{\alpha,\mathrm{ref}}(t_k)}
$$

relatív eltérést képezzük. Az egyezést a rácson vett eltérések nagyságrendje alapján értékeljük: rögzített $N_{\mathrm{boot}}=1000$ (azonosan $N_{\mathrm{sim}}=1000$) mellett akkor tekintjük teljesültnek, ha a rácson vett $|\delta_{\widehat{P}}(t_k)|$ és $|\delta_{q,\alpha}(t_k)|$ eltérések maximális értéke nem haladja meg az $\varepsilon_{\max}=10^{-2}$, és az átlagos értéke nem haladja meg az $\varepsilon_{\mathrm{mean}}=10^{-3}$ küszöböt. A maximális eltéréseket külön riportáljuk, és azok alapján ellenőrizzük, hogy a kilógó pontok nem mutatnak a teljes indexrácsot érintő, tartós mintázatot.

A konzisztencia értelmezéséhez két diagnosztikai ellenőrzést is végzünk:

- az eltérések az indexrács mentén nem mutatnak tartós, egyirányú eltolódást;

- a szimulációs ismétlésszám növelésével ($N_{\mathrm{sim}}$, illetve ahol releváns $N_{\mathrm{boot}}$) a relatív eltérések a nullához konvergálnak, vagyis az indexrácson mért $|\delta_{\widehat{P}}(t_k)|$ és $|\delta_{q,\alpha}(t_k)|$ mennyiségek rendre csökkennek.

## Notebook felépítése

1. Közös paraméterezés rögzítése.  
2. Referencia-kimenet előállítása indexdátumonkénti, egyindexű `prevalence()` hívások sorozatával.  
3. Kiterjesztett implementáció futtatása a teljes indexrácsra.  
4. Célzott összevetés a kimenetszintű mennyiségekre (pontbecslés és intervallumhatárok) az indexrácson.  
5. Konvergenciaellenőrzés: a szimulációs ismétlésszám növelésével ($N_{\mathrm{sim}}$, illetve paraméterben $N_{\mathrm{boot}}$) a referencia--kiterjesztett relatív eltérések nullához tartásának vizsgálata.

---
### 1. Közös paraméter konfiguráció
- A referencia és a kiterjesztett megoldása futtatása ugyanebből a paraméterkészletből dolgozik, így az összevetés konzisztens.

In [1]:
# Common configuration (shared across runs)
seeds <- c(20260301, 20260302, 20260303, 20260304, 20260305, 20260306, 20260307, 20260308, 20260309, 20260310)
seed <- seeds[1]
N_boot <- 1000L
num_years <- c(3, 5, 10, 13, 15, 20, 25)

# Index date grid (K = number of index dates)
K <- 3L
index_start <- as.Date("2012-01-07")
index_dates <- seq.Date(from = index_start, by = "year", length.out = K)

# Model + prevalence settings
inc_formula <- entrydate ~ sex
surv_formula <- survival::Surv(time, status) ~ age + sex
dist <- "weibull"
population_size <- 1e6
death_column <- "eventdate"
status_col <- "status"


### 2. Környezet előkészítése
- Betöltjük a szükséges csomagokat (`rprev`, `survival`, `lubridate`), valamint a `prevsim` példaadatot, hogy a referenciafuttatások egységes környezetben és bemeneten induljanak.
- A betöltött adatokból előállítjuk a futtatásokhoz használt munkatáblát (`base_df`), amelyet a későbbi összevetések során konzisztensen használunk.

In [2]:
# Csomagok betöltése
suppressPackageStartupMessages({
  library(rprev)
  library(survival)
  library(lubridate)
})

# Példaadat betöltése (kiinduló tábla)
data(prevsim, package = "rprev")
base_df <- as.data.frame(prevsim)

# Közös, futtatások között megosztott dátumparaméterek
registry_start_date <- min(base_df$entrydate, na.rm = TRUE)
registry_start_date

Warning message:
"package 'rprev' was built under R version 4.4.3"
Warning message:
"package 'lubridate' was built under R version 4.4.3"


[1] "2003-01-07"

### 3. Referencia `rprev` implementáció futtatása
- A referencia implementáció `prevalence()` függvénye egyetlen indexdátumra ad becslést. Több indexdátum esetén a viszonyítási eredmény indexdátumonként ismételt különálló futtatásokból áll elő.
- A kiértékelést több seed beállítás mellett is elvégezzük.

In [3]:
# K>1 (több indexdátum, ismételt K=1 futások)
# A registry start dátumot a környezet-előkészítő blokkban rögzítettük: `registry_start_date`

ref_abs_table_by_seed <- list()
res_ref_by_index_by_seed <- list()

# Gyors áttekintés: abszolút prevalencia indexdátumonként, horizontonként
extract_abs <- function(res) {
  out <- sapply(num_years, function(y) {
    as.numeric(res$estimates[[paste0("y", y)]][["absolute.prevalence"]])
  })
  names(out) <- paste0("y", num_years)
  out
}

# A seedek listáját a közös konfigurációban adjuk meg: `seeds`
for (s in seeds) {
  cat("\n", paste(rep("-", 80), collapse = ""), "\n", sep = "")
  cat("seed =", s, "\n")

  # Reprodukálható viszonyítás: az indexdátumonkénti referencia-futtatásokat ugyanazzal a seeddel indítjuk,
  # így a K>1 viszonyítási sorozat indexek közötti különbségeit nem a véletlenszám-hívások "elmászó" állapota,
  # hanem az eltérő indexdátumokhoz tartozó becslési helyzet magyarázza.
  res_ref_by_index_s <- lapply(index_dates, function(idx) {
    set.seed(s)
    rprev::prevalence(
      index = idx,
      num_years_to_estimate = num_years,
      data = base_df,
      inc_formula = inc_formula,
      surv_formula = surv_formula,
      dist = dist,
      population_size = population_size,
      death_column = death_column,
      registry_start_date = registry_start_date,
      N_boot = N_boot
    )
  })
  names(res_ref_by_index_s) <- as.character(index_dates)
  res_ref_by_index_by_seed[[as.character(s)]] <- res_ref_by_index_s

  ref_abs_table_s <- t(sapply(res_ref_by_index_s, extract_abs))
  ref_abs_table_by_seed[[as.character(s)]] <- ref_abs_table_s
  print(ref_abs_table_s)

  if (s == seeds[1]) {
    res_ref_by_index <- res_ref_by_index_s
    ref_abs_table <- ref_abs_table_s
  }
}

# A további összevetésekhez a `seeds[1]` futtatás eredményeit használjuk
# ref_abs_table



--------------------------------------------------------------------------------
seed = 20260301 
            y3  y5    y10    y13    y15    y20    y25
2012-01-07 207 297 508.35 598.35 650.83 762.57 850.63
2013-01-07 207 304 517.00 606.37 658.48 769.78 857.76
2014-01-07 127 226 433.00 530.41 582.96 694.63 782.69

--------------------------------------------------------------------------------
seed = 20260302 
            y3  y5    y10   y13    y15    y20    y25
2012-01-07 207 297 508.49 598.4 650.73 762.22 850.62
2013-01-07 207 304 517.00 606.0 658.32 769.98 858.73
2014-01-07 127 226 433.00 530.3 582.69 694.12 782.52

--------------------------------------------------------------------------------
seed = 20260303 
            y3  y5    y10    y13    y15    y20    y25
2012-01-07 207 297 508.26 597.34 650.02 762.02 851.34
2013-01-07 207 304 517.00 606.35 658.66 770.77 860.07
2014-01-07 127 226 433.00 529.76 582.51 694.44 783.76

----------------------------------------------------------

### 4. Kiterjesztett rprev csomag betöltése lokális forrásból és git-meta rögzítése
- A csomag lokális forrásból kerül betöltésre, így a futtatás ténylegesen a vizsgált, módosított implementációval történik.
- A futtatott verzió azonosíthatóságát a git ág és commit azonosító kiírása biztosítja.

In [4]:
# Projekt lokális gyökérkönyvtárának meghatározása (git repo root)
project_root <- trimws(system2("git", c("rev-parse", "--show-toplevel"), stdout = TRUE))
if (length(project_root) == 0 || !dir.exists(project_root)) {
  stop("Could not resolve project root from git.")
}

# Telepített rprev leválasztása, majd lokális forrásból betöltés
if ("package:rprev" %in% search()) detach("package:rprev", unload = TRUE, character.only = TRUE)
if ("rprev" %in% loadedNamespaces()) unloadNamespace("rprev")
if (!requireNamespace("devtools", quietly = TRUE)) stop("devtools package is not available.")

suppressPackageStartupMessages(devtools::load_all(project_root, quiet = TRUE))

# Betöltött csomag elérési útjának megjelenítése
cat(
  "Loaded rprev from: ",
  # normalizePath(getNamespaceInfo("rprev", "path"), winslash = "/"),
  # "\n",
  sep = ""
)

Warning message:
"package 'testthat' was built under R version 4.4.3"


Loaded rprev from: 

In [5]:
# Git meta rögzítése: futtatott kódverzió azonosítása
git_cmd <- function(args) {
  out <- suppressWarnings(
    tryCatch(system2("git", args, stdout = TRUE, stderr = TRUE), error = function(e) character(0))
  )
  status <- attr(out, "status")
  if (!is.null(status) || length(out) == 0) return(NA_character_)
  trimws(out[[1]])
}

old_wd <- getwd()                         # munkakönyvtár mentése
on.exit(setwd(old_wd), add = TRUE)        # visszaállítás a cella végén
setwd(project_root)                       # git parancsok futtatása a repo gyökeréből

branch <- git_cmd(c("rev-parse", "--abbrev-ref", "HEAD"))
commit <- git_cmd(c("rev-parse", "HEAD"))

cat("Git branch: ", branch, "\n", sep = "")
cat("Git commit: ", commit, "\n", sep = "")

Git branch: notebooks/ref-ext-consistency
Git commit: b567147a198e288fd283932d0ab474b439c39d48


### 5. Kiterjesztett `rprev` implementáció futtatása
- Az előző blokkban betöltött lokális implementáció `prevalence()` függvényének futtatása ugyanazzal az indexdátum- és paraméterkészlettel, mint a referenciafuttatásoknál.
- A kiértékelést több seed beállítás mellett is elvégezzük.

In [6]:
years_lbl <- paste0("y", num_years)

ext_abs_table_by_seed <- list()
res_ext_kmulti_by_seed <- list()

# A seedek listáját a közös konfigurációban adjuk meg: `seeds`
for (s in seeds) {
  cat("\n", paste(rep("-", 80), collapse = ""), "\n", sep = "")
  cat("seed =", s, "\n")

  set.seed(s)
  res_ext_kmulti_s <- prevalence(
    index = index_dates,
    num_years_to_estimate = num_years,
    data = base_df,
    inc_formula = inc_formula,
    surv_formula = surv_formula,
    dist = dist,
    population_size = population_size,
    death_column = death_column,
    registry_start_date = registry_start_date,
    N_boot = N_boot
  )
  print(summary(res_ext_kmulti_s))
  res_ext_kmulti_by_seed[[as.character(s)]] <- res_ext_kmulti_s

  ext_abs_table_s <- do.call(
    cbind,
    lapply(num_years, function(y) {
      est <- res_ext_kmulti_s$estimates[[paste0("y", y)]]
      est <- est[order(est$index_date), , drop = FALSE]
      as.numeric(est[["absolute.prevalence"]])
    })
  )
  colnames(ext_abs_table_s) <- years_lbl
  rownames(ext_abs_table_s) <- as.character(sort(unique(res_ext_kmulti_s$index_dates)))
  ext_abs_table_by_seed[[as.character(s)]] <- ext_abs_table_s
  print(ext_abs_table_s)

  if (s == seeds[1]) {
    res_ext_kmulti <- res_ext_kmulti_s
    ext_abs_table <- ext_abs_table_s
  }
}

# A további összevetésekhez a `seeds[1]` futtatás eredményeit használjuk
# ext_abs_table


--------------------------------------------------------------------------------
seed = 20260301 
Prevalence 
~~~~~~~~~~
Estimated prevalence:
 index_date years absolute.prevalence per100K per100K.upper per100K.lower
 2012-01-07     3              207.00   20.70         23.52         17.88
 2012-01-07     5              297.00   29.70         33.08         26.32
 2012-01-07    10              508.03   50.80         56.20         45.41
 2012-01-07    13              597.82   59.78         66.64         52.93
 2012-01-07    15              650.47   65.05         72.65         57.44
 2012-01-07    20              762.66   76.27         85.56         66.97
 2012-01-07    25              851.23   85.12         95.92         74.33
 2013-01-07     3              207.00   20.70         23.52         17.88
 2013-01-07     5              304.00   30.40         33.82         26.98
 2013-01-07    10              517.00   51.70         56.16         47.24
 2013-01-07    13              606.39   60

### 6. Eredmények összehasonlítása (referencia vs kiterjesztett)
- Az alábbi táblázat az abszolút prevalencia-becsléseket veti össze indexdátumonként és időhorizontonként.
- A `counted` oszlop a regiszterből közvetlenül számlált prevalens esetszámot mutatja (indexdátumonként), a `ref` és `ext` oszlopok pedig az eljárások becsléseit.
- A `delta` oszlop a két becslés különbsége: `delta = kiterjesztett - referencia`.
- A relatív eltérést a `delta_rel = (ext - ref) / ref` és `abs_delta_rel = |delta_rel|` oszlopok adják.
- A `eps_ok_max` és `eps_ok_mean` jelzi, hogy az adott rácsponton `abs_delta_rel` rendre `1e-2`, illetve `1e-3` alatt marad-e.


In [7]:
# Dinamikus összevetés: referencia (ismételt K=1) vs kiterjesztett (natív többindexű), seed szerint

counted_by_date <- sapply(res_ref_by_index, function(res) as.numeric(res$counted))
names(counted_by_date) <- names(res_ref_by_index)

epsilon_max <- 1e-2
epsilon_mean <- 1e-3

build_compare_df <- function(ref_mat, ext_mat) {
  idx <- intersect(rownames(ref_mat), rownames(ext_mat))
  yrs <- intersect(colnames(ref_mat), colnames(ext_mat))

  df <- expand.grid(index_date = idx, horizon = yrs, stringsAsFactors = FALSE)
  df$ref <- mapply(function(r, c) as.numeric(ref_mat[r, c]), df$index_date, df$horizon)
  df$ext <- mapply(function(r, c) as.numeric(ext_mat[r, c]), df$index_date, df$horizon)

  counted_i <- counted_by_date[as.character(idx)]
  df$counted <- as.numeric(counted_i[df$index_date])

  df$delta <- df$ext - df$ref
  df$delta_rel <- ifelse(df$ref == 0, NA_real_, df$delta / df$ref)
  df$abs_delta_rel <- abs(df$delta_rel)
  df$eps_ok_max <- df$abs_delta_rel <= epsilon_max
  df$eps_ok_mean <- df$abs_delta_rel <= epsilon_mean

  df$horizon_years <- as.integer(sub("^y", "", df$horizon))
  idx_d <- as.Date(df$index_date)
  start_d <- idx_d - round(df$horizon_years * 365.25)
  yrs_need <- as.numeric(difftime(registry_start_date, start_d, units = "days")) / 365.25
  df$sim_inc_years <- as.integer(round(pmax(0, pmin(df$horizon_years, yrs_need))))
  df <- df[order(as.Date(df$index_date), df$horizon_years), ]
  df$horizon <- NULL
  df <- df[, c("index_date", "horizon_years", "sim_inc_years", "counted", "ref", "ext", "delta", "delta_rel", "abs_delta_rel", "eps_ok_max", "eps_ok_mean")]

  df$ref <- round(df$ref, 2)
  df$ext <- round(df$ext, 2)
  df$delta <- round(df$delta, 2)
  df$delta_rel <- round(df$delta_rel, 6)
  df$abs_delta_rel <- round(df$abs_delta_rel, 6)

  df
}

compare_list <- lapply(seeds, function(s) {
  ref_mat <- ref_abs_table_by_seed[[as.character(s)]]
  ext_mat <- ext_abs_table_by_seed[[as.character(s)]]
  df <- build_compare_df(ref_mat, ext_mat)
  df$seed <- s
  df
})

compare_all <- data.table::rbindlist(compare_list, fill = TRUE)
compare_all <- compare_all[order(compare_all$seed, as.Date(compare_all$index_date), compare_all$horizon_years), ]
compare_all <- compare_all[, c("seed", "index_date", "horizon_years", "sim_inc_years", "counted", "ref", "ext", "delta", "delta_rel", "abs_delta_rel", "eps_ok_max", "eps_ok_mean")]

compare_all


seed,index_date,horizon_years,sim_inc_years,counted,ref,ext,delta,delta_rel,abs_delta_rel,eps_ok_max,eps_ok_mean
<dbl>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>
20260301,2012-01-07,3,0,475,207.00,207.00,0.00,0.000000,0.000000,TRUE,TRUE
20260301,2012-01-07,5,0,475,297.00,297.00,0.00,0.000000,0.000000,TRUE,TRUE
20260301,2012-01-07,10,1,475,508.35,508.03,-0.32,-0.000629,0.000629,TRUE,TRUE
20260301,2012-01-07,13,4,475,598.35,597.82,-0.53,-0.000886,0.000886,TRUE,TRUE
20260301,2012-01-07,15,6,475,650.83,650.47,-0.36,-0.000553,0.000553,TRUE,TRUE
20260301,2012-01-07,20,11,475,762.57,762.66,0.09,0.000118,0.000118,TRUE,TRUE
20260301,2012-01-07,25,16,475,850.63,851.23,0.60,0.000705,0.000705,TRUE,TRUE
20260301,2013-01-07,3,0,517,207.00,207.00,0.00,0.000000,0.000000,TRUE,TRUE
20260301,2013-01-07,5,0,517,304.00,304.00,0.00,0.000000,0.000000,TRUE,TRUE


#### 7. Konfidencia-intervallumok összevetése (referencia vs kiterjesztett)
- A pontbecslések mellett a prevalencia-arányra számított konfidencia-intervallumok alsó és felső határát is összevetjük a két megoldás között.
- A relatív eltérést külön képezzük az alsó és a felső határra: `delta_rel = (q_ext - q_ref) / q_ref`, illetve `abs_delta_rel = |delta_rel|`.
- Az `eps_ok_max_*` és `eps_ok_mean_*` jelzi, hogy az adott rácsponton `abs_delta_rel` rendre `1e-2`, illetve `1e-3` alatt marad-e (külön az alsó és külön a felső határra).

In [8]:
# Konfidencia-intervallumok (alsó/felső) összevetése a korábbi futások eredményeiből

stopifnot(exists("res_ref_by_index_by_seed"), exists("res_ext_kmulti_by_seed"))

get_ci_cols <- function(nms) {
  lower <- grep("^per.*\\.lower$", nms, value = TRUE)
  upper <- grep("^per.*\\.upper$", nms, value = TRUE)
  stopifnot(length(lower) >= 1, length(upper) >= 1)
  list(lower = lower[[1]], upper = upper[[1]])
}

parse_denom_from_per <- function(per_lab) {
  # pl. per100K -> 100000, per1M -> 1000000
  x <- sub("^per", "", per_lab)
  m <- regexec("^([0-9]+(?:\\.[0-9]+)?)([KM]?)$", x)
  g <- regmatches(x, m)[[1]]
  stopifnot(length(g) > 0)
  val <- as.numeric(g[2])
  unit <- g[3]
  mult <- if (unit == "K") 1e3 else if (unit == "M") 1e6 else 1
  val * mult
}

to_abs <- function(rate, denom, pop) {
  as.numeric(rate) / as.numeric(denom) * as.numeric(pop)
}

extract_ref_ci_mat <- function(res_ref_by_index_s, ci_col, denom) {
  mat <- t(sapply(res_ref_by_index_s, function(res) {
    sapply(num_years, function(y) {
      est <- res$estimates[[paste0("y", y)]]
      to_abs(est[[ci_col]], denom, population_size)
    })
  }))
  colnames(mat) <- paste0("y", num_years)
  mat
}

extract_ext_ci_mat <- function(res_ext_kmulti_s, ci_col, denom) {
  mat <- do.call(cbind, lapply(num_years, function(y) {
    est <- res_ext_kmulti_s$estimates[[paste0("y", y)]]
    est <- est[order(est$index_date), , drop = FALSE]
    to_abs(est[[ci_col]], denom, population_size)
  }))
  colnames(mat) <- paste0("y", num_years)
  rownames(mat) <- as.character(sort(unique(res_ext_kmulti_s$index_dates)))
  mat
}

build_compare_ci_df <- function(ref_lower, ref_upper, ext_lower, ext_upper) {
  idx <- intersect(rownames(ref_lower), rownames(ext_lower))
  yrs <- intersect(colnames(ref_lower), colnames(ext_lower))

  df <- expand.grid(index_date = idx, horizon = yrs, stringsAsFactors = FALSE)
  df$ref_lower <- mapply(function(r, c) as.numeric(ref_lower[r, c]), df$index_date, df$horizon)
  df$ext_lower <- mapply(function(r, c) as.numeric(ext_lower[r, c]), df$index_date, df$horizon)
  df$ref_upper <- mapply(function(r, c) as.numeric(ref_upper[r, c]), df$index_date, df$horizon)
  df$ext_upper <- mapply(function(r, c) as.numeric(ext_upper[r, c]), df$index_date, df$horizon)

  df$delta_lower <- df$ext_lower - df$ref_lower
  df$delta_upper <- df$ext_upper - df$ref_upper

  df$delta_rel_lower <- ifelse(df$ref_lower == 0, NA_real_, df$delta_lower / df$ref_lower)
  df$delta_rel_upper <- ifelse(df$ref_upper == 0, NA_real_, df$delta_upper / df$ref_upper)

  df$abs_delta_rel_lower <- abs(df$delta_rel_lower)
  df$abs_delta_rel_upper <- abs(df$delta_rel_upper)


  df$horizon_years <- as.integer(sub("^y", "", df$horizon))
  idx_d <- as.Date(df$index_date)
  start_d <- idx_d - round(df$horizon_years * 365.25)
  yrs_need <- as.numeric(difftime(registry_start_date, start_d, units = "days")) / 365.25
  df$sim_inc_years <- as.integer(round(pmax(0, pmin(df$horizon_years, yrs_need))))
  df <- df[order(as.Date(df$index_date), df$horizon_years), ]
  df$horizon <- NULL

  df <- df[, c(
    "index_date", "horizon_years", "sim_inc_years",
    "ref_lower", "ext_lower", "delta_lower", "delta_rel_lower", "abs_delta_rel_lower",
    "ref_upper", "ext_upper", "delta_upper", "delta_rel_upper", "abs_delta_rel_upper"
  )]

  df$ref_lower <- round(df$ref_lower, 2)
  df$ext_lower <- round(df$ext_lower, 2)
  df$delta_lower <- round(df$delta_lower, 2)
  df$delta_rel_lower <- round(df$delta_rel_lower, 6)
  df$abs_delta_rel_lower <- round(df$abs_delta_rel_lower, 6)

  df$ref_upper <- round(df$ref_upper, 2)
  df$ext_upper <- round(df$ext_upper, 2)
  df$delta_upper <- round(df$delta_upper, 2)
  df$delta_rel_upper <- round(df$delta_rel_upper, 6)
  df$abs_delta_rel_upper <- round(df$abs_delta_rel_upper, 6)
  df$sim_inc_years <- as.integer(df$sim_inc_years)

  data.table::as.data.table(df)
}

compare_ci_by_seed <- lapply(seeds, function(s) {
  res_ref_by_index_s <- res_ref_by_index_by_seed[[as.character(s)]]
  res_ext_kmulti_s <- res_ext_kmulti_by_seed[[as.character(s)]]

  # CI oszlopnév + denominátor a referencia-kimenetből
  y0 <- num_years[[1]]
  est0 <- res_ref_by_index_s[[1]]$estimates[[paste0("y", y0)]]
  ci_cols <- get_ci_cols(names(est0))
  per_lab <- sub("\\.lower$", "", ci_cols$lower)
  denom <- parse_denom_from_per(per_lab)

  ref_lower <- extract_ref_ci_mat(res_ref_by_index_s, ci_cols$lower, denom)
  ref_upper <- extract_ref_ci_mat(res_ref_by_index_s, ci_cols$upper, denom)

  ext_lower <- extract_ext_ci_mat(res_ext_kmulti_s, ci_cols$lower, denom)
  ext_upper <- extract_ext_ci_mat(res_ext_kmulti_s, ci_cols$upper, denom)

  out <- build_compare_ci_df(ref_lower, ref_upper, ext_lower, ext_upper)
  out[, seed := s]
  out <- out[, c("seed", setdiff(names(out), "seed")), with = FALSE]
  out
})

compare_ci_all <- data.table::rbindlist(compare_ci_by_seed, use.names = TRUE, fill = TRUE)
compare_ci_all


seed,index_date,horizon_years,sim_inc_years,ref_lower,ext_lower,delta_lower,delta_rel_lower,abs_delta_rel_lower,ref_upper,ext_upper,delta_upper,delta_rel_upper,abs_delta_rel_upper
<dbl>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
20260301,2012-01-07,3,0,178.8,178.8,0.0,0.000000,0.000000,235.2,235.2,0.0,0.000000,0.000000
20260301,2012-01-07,5,0,263.2,263.2,0.0,0.000000,0.000000,330.8,330.8,0.0,0.000000,0.000000
20260301,2012-01-07,10,1,454.1,454.1,0.0,0.000000,0.000000,562.6,562.0,-0.6,-0.001066,0.001066
20260301,2012-01-07,13,4,529.9,529.3,-0.6,-0.001132,0.001132,666.8,666.4,-0.4,-0.000600,0.000600
20260301,2012-01-07,15,6,574.6,574.4,-0.2,-0.000348,0.000348,727.1,726.5,-0.6,-0.000825,0.000825
20260301,2012-01-07,20,11,670.9,669.7,-1.2,-0.001789,0.001789,854.3,855.6,1.3,0.001522,0.001522
20260301,2012-01-07,25,16,742.9,743.3,0.4,0.000538,0.000538,958.4,959.2,0.8,0.000835,0.000835
20260301,2013-01-07,3,0,178.8,178.8,0.0,0.000000,0.000000,235.2,235.2,0.0,0.000000,0.000000
20260301,2013-01-07,5,0,269.8,269.8,0.0,0.000000,0.000000,338.2,338.2,0.0,0.000000,0.000000


#### 8. Különbségek összefoglalása

Az alábbi összesítő táblázat a seedek mentén előállított összehasonlító táblák (`compare_all`) alapján, indexdátumonként és horizontonként a relatív eltérés (`|delta_rel|`) leíró statisztikáit adja meg.

- A sorok egy-egy indexdátum–horizont párt jelentenek.
- Az oszlopok a seed-választásból adódó ingadozás mértékét írják le (`min`, `max`, `median`, `mean`, `sd`), valamint megjelenítik a küszöböket és a hozzájuk tartozó megfelelést.
- A `sim_inc_years` oszlop azt mutatja, hogy az adott indexdátum–horizont párhoz a `registry_start_date`-ig hány évnyi incidenciát kell szimulálni (0, ha a teljes időablak a regiszteridőszakon belül van).

Ez a táblázat közvetlen viszonyítási alapot ad annak megítéléséhez, hogy a referencia és a kiterjesztett megoldás közötti eltérés nagyságrendje mennyiben illeszkedik a seed-függő változékonyság tartományába.


In [9]:
# Összefoglalás: |delta| leíró statisztikák seedek mentén

cmp <- data.table::as.data.table(compare_all)

if (!"abs_delta_rel" %in% names(cmp)) {
  cmp[, delta_rel := (ext - ref) / ref]
  cmp[, abs_delta_rel := abs(delta_rel)]
}

summary_abs_delta <- cmp[
  , .(
    ref = {
      v <- ref[seed == seeds[1]]
      if (length(v) == 0) v <- ref
      as.numeric(v[1])
    },
    sim_inc_years = {
      idx_d <- as.Date(index_date[1])
      start_d <- idx_d - round(horizon_years[1] * 365.25)
      yrs_need <- as.numeric(difftime(registry_start_date, start_d, units = "days")) / 365.25
      as.integer(round(pmax(0, pmin(horizon_years[1], yrs_need))))
    },
    abs_delta_rel_min = min(abs_delta_rel, na.rm = TRUE),
    abs_delta_rel_max = max(abs_delta_rel, na.rm = TRUE),
    abs_delta_rel_med = stats::median(abs_delta_rel, na.rm = TRUE),
    abs_delta_rel_mean = mean(abs_delta_rel, na.rm = TRUE),
    abs_delta_rel_sd = stats::sd(abs_delta_rel, na.rm = TRUE)
  ),
  by = .(index_date, horizon_years)
]

summary_abs_delta <- summary_abs_delta[order(as.Date(index_date), horizon_years)]
summary_abs_delta <- summary_abs_delta[, c(
  "index_date", "horizon_years", "sim_inc_years", "ref",
  "abs_delta_rel_min", "abs_delta_rel_max", "abs_delta_rel_med", "abs_delta_rel_mean", "abs_delta_rel_sd"
), with = FALSE]

summary_abs_delta[, `:=`(
  ref = round(ref, 2),
  sim_inc_years = as.integer(sim_inc_years),
  abs_delta_rel_min = round(abs_delta_rel_min, 6),
  abs_delta_rel_max = round(abs_delta_rel_max, 6),
  abs_delta_rel_med = round(abs_delta_rel_med, 6),
  abs_delta_rel_mean = round(abs_delta_rel_mean, 6),
  abs_delta_rel_sd = round(abs_delta_rel_sd, 6)
)]

summary_abs_delta


index_date,horizon_years,sim_inc_years,ref,abs_delta_rel_min,abs_delta_rel_max,abs_delta_rel_med,abs_delta_rel_mean,abs_delta_rel_sd
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2012-01-07,3,0,207.00,0.000000,0.000000,0.000000,0.000000,0.000000
2012-01-07,5,0,297.00,0.000000,0.000000,0.000000,0.000000,0.000000
2012-01-07,10,1,508.35,0.000000,0.000748,0.000207,0.000301,0.000263
2012-01-07,13,4,598.35,0.000084,0.001523,0.000527,0.000628,0.000508
2012-01-07,15,6,650.83,0.000246,0.002557,0.000530,0.000815,0.000686
2012-01-07,20,11,762.57,0.000053,0.002393,0.000945,0.000922,0.000848
2012-01-07,25,16,850.63,0.000224,0.002972,0.001116,0.001159,0.000788
2013-01-07,3,0,207.00,0.000000,0.000000,0.000000,0.000000,0.000000
2013-01-07,5,0,304.00,0.000000,0.000000,0.000000,0.000000,0.000000


### 9. Összesítés (egyezési kritériumok)

Az alábbi összefoglaló táblák a bevezetőben és a dolgozat validálási kritériumai között rögzített relatív eltérésre épülnek. A pontbecslésre és a konfidencia-intervallumok határaira rendre:

- `delta_P = (P_ext - P_ref) / P_ref`,
- `delta_q = (q_ext - q_ref) / q_ref`, `alpha` értékekre `0.025` és `0.975`.

Az egyezést a rácson és seedenként képzett `|delta|` eltérések maximális és átlagos értékei alapján értékeljük. A küszöbök: `epsilon_max = 1e-2` a maximális eltérésre, illetve `epsilon_mean = 1e-3` az átlagos eltérésre.

In [10]:
# Összesítés: relatív eltérések (pontbecslés és intervallumhatárok)
epsilon_max <- 1e-2
epsilon_mean <- 1e-3

stopifnot(exists("compare_all"), exists("compare_ci_all"))

# 1) Pontbecslések: |delta_P|
cmp_est <- data.table::as.data.table(compare_all)
cmp_est[, delta_rel := ifelse(ref == 0, NA_real_, (ext - ref) / ref)]
cmp_est[, abs_delta_rel := abs(delta_rel)]

aggregate_abs_delta <- function(dt, case_label, by_cols = NULL) {
  if (is.null(by_cols) || length(by_cols) == 0) {
    out <- dt[, .(
      abs_delta_rel_max = max(abs_delta_rel, na.rm = TRUE),
      abs_delta_rel_mean = mean(abs_delta_rel, na.rm = TRUE),
      epsilon_max = epsilon_max,
      epsilon_mean = epsilon_mean
    )]
  } else {
    out <- dt[, .(
      abs_delta_rel_max = max(abs_delta_rel, na.rm = TRUE),
      abs_delta_rel_mean = mean(abs_delta_rel, na.rm = TRUE),
      epsilon_max = epsilon_max,
      epsilon_mean = epsilon_mean
    ), by = by_cols]
  }
  out[, case := case_label]
  out
}

summary_est <- data.table::rbindlist(list(
  aggregate_abs_delta(cmp_est, case_label = "all"),
  aggregate_abs_delta(cmp_est[as.integer(sim_inc_years) == 10], case_label = "sim_inc_years_int10")
), use.names = TRUE, fill = TRUE)
summary_est <- summary_est[order(case)]
summary_est[, `:=`(
  eps_ok_max = abs_delta_rel_max <= epsilon_max,
  eps_ok_mean = abs_delta_rel_mean <= epsilon_mean,
  eps_ok = (abs_delta_rel_max <= epsilon_max) & (abs_delta_rel_mean <= epsilon_mean)
)]

summary_est[, `:=`(
  abs_delta_rel_max = round(abs_delta_rel_max, 6),
  abs_delta_rel_mean = round(abs_delta_rel_mean, 6)
)]

# 2) Intervallumhatárok: |delta_q,alpha| (alpha = 0.025, 0.975)
cmp_ci <- data.table::as.data.table(compare_ci_all)
cmp_ci[, delta_rel_lower := ifelse(ref_lower == 0, NA_real_, (ext_lower - ref_lower) / ref_lower)]
cmp_ci[, delta_rel_upper := ifelse(ref_upper == 0, NA_real_, (ext_upper - ref_upper) / ref_upper)]

ci_long <- data.table::rbindlist(list(
  cmp_ci[, .(alpha = 0.025, abs_delta_rel = abs(delta_rel_lower), sim_inc_years = as.integer(sim_inc_years))],
  cmp_ci[, .(alpha = 0.975, abs_delta_rel = abs(delta_rel_upper), sim_inc_years = as.integer(sim_inc_years))]
), use.names = TRUE, fill = TRUE)

summary_ci <- data.table::rbindlist(list(
  aggregate_abs_delta(ci_long, case_label = "all", by_cols = "alpha"),
  aggregate_abs_delta(ci_long[as.integer(sim_inc_years) == 10], case_label = "sim_inc_years_int10", by_cols = "alpha")
), use.names = TRUE, fill = TRUE)
summary_ci <- summary_ci[order(alpha, case)]
summary_ci[, `:=`(
  eps_ok_max = abs_delta_rel_max <= epsilon_max,
  eps_ok_mean = abs_delta_rel_mean <= epsilon_mean,
  eps_ok = (abs_delta_rel_max <= epsilon_max) & (abs_delta_rel_mean <= epsilon_mean)
)]

summary_ci[, `:=`(
  abs_delta_rel_max = round(abs_delta_rel_max, 6),
  abs_delta_rel_mean = round(abs_delta_rel_mean, 6)
)]

summary_est
summary_ci


abs_delta_rel_max,abs_delta_rel_mean,epsilon_max,epsilon_mean,case,eps_ok_max,eps_ok_mean,eps_ok
<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<lgl>,<lgl>,<lgl>
0.003394,0.000527,0.01,0.001,all,TRUE,TRUE,TRUE
0.001868,0.001134,0.01,0.001,sim_inc_years_int10,TRUE,FALSE,FALSE


alpha,abs_delta_rel_max,abs_delta_rel_mean,epsilon_max,epsilon_mean,case,eps_ok_max,eps_ok_mean,eps_ok
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<lgl>,<lgl>,<lgl>
0.025,0.006299,0.001035,0.01,0.001,all,TRUE,FALSE,FALSE
0.025,0.002494,0.001204,0.01,0.001,sim_inc_years_int10,TRUE,FALSE,FALSE
0.975,0.006154,0.001030,0.01,0.001,all,TRUE,FALSE,FALSE
0.975,0.003598,0.001732,0.01,0.001,sim_inc_years_int10,TRUE,FALSE,FALSE


### 10. Megállapítás

A fenti prefix-tábla oszlopaiban ugyanazt az eljárást futtatjuk eltérő horizontkészletekkel (a `num_years` vektor prefixeivel), miközben minden futtatás előtt ugyanazt a véletlenszám-generátor magot (seedet) állítjuk be. Ennek ellenére a közös horizontokra kapott becslések nem feltétlenül egyeznek, mert a horizontkészlet megváltoztatása a szimuláció teljes lefutását is megváltoztatja.

A fő ok a véletlenszám-felhasználás eltérő ütemezése és mennyisége:

- A `prevalence()` a szimuláció induló dátumát a **legnagyobb kért horizont** alapján állítja be: `sim_start_date <- min(index_dates) - years(max(num_years_to_estimate))` (R/prevalence.R). Ha a horizontlistát bővítjük, a `sim_start_date` korábbra kerül.
- A `sim_prevalence()` ezután a szimulációs ablak hosszát a `starting_date` és a legkésőbbi indexdátum különbségéből számolja: `number_incident_days <- difftime(max(index_dates), starting_date, ...)` (R/prevalence.R). A hosszabb ablak **más (és tipikusan több) szimulált incidens esetet** eredményezhet a `draw_incident_population(...)` hívásban, ami a véletlenszám-generátor állapotát is másképp "fogyasztja".
- A kiterjesztett Monte Carlo magban futásonként egyszer sorsolunk esetszintű küszöböt: `xi <- runif(nrow(incident_population))`, majd ezt az összes indexdátumhoz felhasználjuk: `alive_k <- I(xi <= p_k)` (R/prevalence.R). Ha a szimulált incidens populáció mérete (és így a `xi` hossza) megváltozik, akkor ugyanazon kezdő seed mellett is eltérő RNG-állapotból folytatódnak a további lépések.

A referencia megoldáshoz képest a kiterjesztés a státuszképzés véletlen komponensét is másképp szervezi: a túlélési valószínűségekhez tartozó bináris státusz nem független Bernoulli-sorsolások sorozataként áll elő, hanem az esetszintű `xi ~ Unif(0,1)` küszöb és a `p_k` túlélési valószínűségek összehasonlításával. Ez azonos seed használat mellett is megváltoztatja a véletlenszám-hívások mintázatát, ezért ugyanazon véletlenmag mellett sem várható szigorú numerikus egyezés.

Módszertani szempontból a kiterjesztés célja az, hogy **egy futáson belül** az indexdátumok mentén időben konzisztens státuszképzés történjen (dolgozat 5. fejezet: esetszintű `xi` küszöb és a Delta-mátrix/maszk konstrukció). Ez a tulajdonság nem garantálja, hogy **külön futtatások** (eltérő horizontkészletek) között a közös horizontokra kapott becslések változatlanok maradnak.

Gyakorlati értelmezés: a prefix-oszlopok közötti különbség lényegében azt jelenti, hogy a közös horizont becslése egy másik (ugyanazzal a maggal indított, de másképp felhasznált) véletlenszám-sorozaton alapul. Emiatt a különbségek nagyságrendje tipikusan összemérhető azzal a variabilitással, amit különböző seed-beállítások is okoznának.

További megállapítás a teljes notebook alapján: a seedek mentén képzett összevető táblák összesítése (az indexdátum–horizont párokhoz tartozó `|delta|` leíró statisztikái; lásd az "Összefoglalás (seed-variabilitás az eltérésben)" táblázatot, `summary_abs_delta`) azt mutatja, hogy a referencia és a kiterjesztett becslések közötti eltérések nagyságrendje a vizsgált beállítások mellett jellemzően a seed-választásból adódó Monte Carlo változékonysággal összemérhető. A különbségek előjele seed szerint is változhat, és az indexdátumok mentén nem rajzolódik ki tartós, egyirányú eltolódás; ez összhangban áll azzal, hogy a megfigyelt eltérések elsődleges forrása a futási szerkezet és a véletlenszám-felhasználás ütemezésének különbsége.

Mindezek alapján a kiterjesztett eljárás által adott becslések a referencia-megoldás eredményeivel **összevethetők**, és a megfigyelt eltérések a vizsgált beállítások mellett **módszertanilag elfogadható mértékűeknek** tekinthetők. Ennek megfelelően a kiterjesztett implementáció a referencia-eljárás logikájával **konzisztens** kimenetet szolgáltat, és a több indexdátum együttes kezelésére vonatkozó kiterjesztés validáltnak tekinthető a jelen vizsgálati keretben.


# todo
1. a kiértékelő táblákat erre a epsilon alapú különbségkre kell átvariálni
2. a konfidencia intervallumokat ugyanígy ki kell értékelni
3. Meg kell vizsgálni, hogy konvergál-e a nullához, ha növelem az ismétlésszámot
4. Ennek fényében átírni a megállapításokat